# ME BDO Master Data Raw Ingestion

Loads master data parquet files from the QA landing volume into Delta tables in `bci_rbs_dev.raw`, prefixed with `me_bdo_`.

In [0]:
import os, re

# Parameters
source_path = "/Volumes/bci_rbs_qa/raw/landing_meb_volume/masterData/organization"
target_schema = "bci_rbs_dev.raw"
table_prefix = "me_bdo_"

def list_files(path):
    """Recursively list files below path; skip markers like _SUCCESS / .crc."""
    files = []
    for f in dbutils.fs.ls(path):
        if f.name.startswith(("_", ".")):
            continue
        if f.isDir():
            files.extend(list_files(f.path))
        else:
            files.append(f.path)
    return files

def table_name_for(file_path):
    """me_bdo_<file name without extension>, lowercase, only [a-z0-9_]."""
    stem = os.path.splitext(os.path.basename(file_path))[0].lower()
    return f"{target_schema}.{table_prefix}{re.sub(r'[^a-z0-9_]', '_', stem)}"

source_files = list_files(source_path)
print(f"Found {len(source_files)} files in {source_path}")

# Read each parquet file and write it as a managed Delta table
results = []
for file_path in source_files:
    target_table = table_name_for(file_path)
    try:
        (spark.read.parquet(file_path)
             .write.mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(target_table))
        results.append((file_path, target_table, "OK", spark.table(target_table).count(), None))
    except Exception as e:
        results.append((file_path, target_table, "FAILED", None, str(e)[:300]))

summary_df = spark.createDataFrame(
    results, "source_file string, target_table string, status string, row_count long, error string")
display(summary_df)
assert all(r[2] == "OK" for r in results), "Some files failed to load - see summary above"

In [0]:
import os, re

# Parameters
excel_path = "/Volumes/bci_rbs_qa/raw/landing_meb_volume/reportData/OperatorSelfCheck/CS-E_vulnmgmt.xlsx"
excel_target_schema = "bci_rbs_dev.raw"
excel_table_prefix = "me_bdo_"
header_rows = 1  # first row holds the column names

def clean_name(name):
    """Lowercase, only [a-z0-9_] (Delta does not allow spaces in column names)."""
    return re.sub(r"_+", "_", re.sub(r"[^a-z0-9_]", "_", name.strip().lower())).strip("_")

stem = os.path.splitext(os.path.basename(excel_path))[0]
excel_target_table = f"{excel_target_schema}.{excel_table_prefix}{clean_name(stem)}"

# Native Databricks Excel reader (no openpyxl / pandas needed); reads the first sheet
excel_df = (spark.read.format("excel")
                .option("headerRows", header_rows)
                .load(excel_path))
excel_df = excel_df.toDF(*[clean_name(c) for c in excel_df.columns])

(excel_df.write.mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(excel_target_table))

print(f"Loaded {excel_path} -> {excel_target_table}: {spark.table(excel_target_table).count()} rows")
display(spark.table(excel_target_table).limit(10))